In [ ]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# %matplotlib qt
# %matplotlib widget

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from dashsrc.plot_components.plot_wrappers.data_selection import group_filter_data

from analytics_processing.modality_loading import session_modality_from_nas
from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames


In [ ]:
%matplotlib widget

In [ ]:
output_dir = "/mnt/SpatialSequenceLearning/Simon/hmm_actions"
data = {}
nas_dir = C.device_paths()[0]
Logger().init_logger(None, None, logging_level="INFO")


In [ ]:
animal_ids = [6]
paradigm_ids = [1100]
session_ids = None

In [ ]:
session_dirs = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)[0]
session_names = fullfnames2snames(session_dirs)

In [ ]:
framew_beh = analytics.get_analytics('BehaviorFramewise', session_names=session_names)#[:34],)
# framew_beh = analytics.get_analytics('BehaviorTrackwise', session_names=session_names)#[:34],)
framew_beh

In [ ]:
framew_beh.columns.tolist()

In [ ]:
action_cols = [
               # forward velocity and acc
               'frame_raw', 
               'frame_raw_500msMedian', 
               'frame_yaw', 
               'frame_pitch', 
               'frame_raw_abs_acc_500msMedian', 
               # off rotations velocity
               'frame_YawPitch_abs_vel_sum_500msMedian', 
               'frame_YawPitch_abs_acc_sum_500msMedian', 
               # sum for reward threshold and acc
               'frame_RawYawPitch_abs_vel_sum',
               'frame_RawYawPitch_abs_vel_sum_500msMedian',
               'frame_RawYawPitch_abs_acc_sum_500msMedian',
               # others
               'lick_detected', 
               'trial_id', 
               'trial_outcome', 
               'choice_R1', 
               # derived metrics
               'frame_forward_prop',
               'velocity_threshold_at_R1',
               'forward_vs_rotation_corr',
               
               'frame_below_velocity_thr'
               ]

# renamer = {
#       'frame_raw_500msMedian': 'ball_forward_vel',
#       'frame_raw_abs_acc_500msMedian': 'ball_forward_acc',
#       'frame_YawPitch_abs_vel_sum_500msMedian': 'ball_rotation_vel',
#       'frame_YawPitch_abs_acc_sum_500msMedian': 'ball_rotation_acc',
#       'frame_RawYawPitch_abs_vel_sum': 'ball_forward+rotation_vel',
#       # 'frame_RawYawPitch_abs_vel_sum_500msMedian': 'ball_forward+rotation_vel',
#       'frame_RawYawPitch_abs_acc_sum_500msMedian': 'ball_forward+rotation_acc',
#       'lick_detected': 'licking',
#       'frame_forward_prop': 'forward_proportion',
#       'forward_vs_rotation_corr': 'forward_vs_rotation_corr',
#       'frame_below_velocity_thr': 'below_r_thr',
# }
framew_beh['frame_below_velocity_thr'] = framew_beh['frame_below_velocity_thr'].fillna(0)
actions = framew_beh[action_cols].dropna() # # just for NaN dropping (first ~80 frames of a session)
actions = actions#.rename(columns=renamer)

actions

In [ ]:
actions_cp = actions.copy()
actions_cp['track_zone'] = framew_beh.loc[actions.index, 'track_zone']
actions_cp['trial_id'] = framew_beh.loc[actions.index, 'trial_id']
actions_cp.reset_index(inplace=True)
actions_cp

nframes_pre, nframes_post = 65, 125
def around_r1_entries(group):
    r1_idx = group[group['track_zone'] == 'reward1Zone'].index[0]
    return np.arange(r1_idx - nframes_pre, r1_idx + nframes_post)
    # r1_idx = group[group['track_zone'] == 'reward1Zone'].index
    # return np.arange(r1_idx[0]-60, r1_idx[-1]+30)

r1_entries = actions_cp.groupby(by=['session_id', 'trial_id']).apply(around_r1_entries, )

r1_entries = np.concatenate(r1_entries.values)
print(r1_entries.shape)
# print(frame_id_from_start.shape)
print(f"Slicing to {nframes_pre} frames before and {nframes_post} frames after R1 entry gives us {len(r1_entries)}/{len(actions_cp)} frames to analyze around R1 entries.")

actions = actions.iloc[r1_entries]
actions['frame_id_from_start'] = actions.groupby(by=['session_id', 'trial_id']).cumcount()

In [ ]:
actions['trial_id']

In [ ]:
actions = actions.set_index(['trial_id', 'frame_id_from_start'], append=True).reset_index(level=[0,1,3])
actions

In [ ]:

# saVE
# final_labels.to_csv(os.path.join(output_dir, "trial_cluster_labels.csv"), index=False)
# print(f"✓ Saved trial cluster labels to {output_dir}/trial_cluster_labels.csv")

hmm_labels = pd.read_csv(os.path.join(output_dir, "trial_cluster_labels.csv"))
hmm_labels = hmm_labels.set_index(['trial_id', 'session_id'], drop=True)
hmm_labels

In [ ]:
plt.close("all")

def plot_all_sessions_vel_props_fast(
    all_data,
    hmm_labels_df,
    target_cluster=3,
    y_col="frame_raw_500msMedian",
    color_col="frame_forward_prop",
    cmap="RdBu_r",
    color_vmin=0.3,
    color_vmax=0.7,
    x_lim=(0, 180),
    y_lim=(0, 150),
):
    # --- Validate inputs ---
    if "cluster_k_id" not in hmm_labels_df.columns:
        raise ValueError("hmm_labels_df must contain 'cluster_k_id' column.")
    if not {"trial_id", "session_id"}.issubset(set(hmm_labels_df.index.names)):
        raise ValueError("hmm_labels_df index must include 'trial_id' and 'session_id'.")
    if y_col not in all_data.columns:
        raise ValueError(f"'{y_col}' not found in all_data columns.")
    if color_col not in all_data.columns:
        raise ValueError(f"'{color_col}' not found in all_data columns.")

    wanted = hmm_labels_df[hmm_labels_df["cluster_k_id"] == target_cluster]
    if wanted.empty:
        print(f"No trials found for cluster {target_cluster}.")
        return

    wanted_pairs = pd.MultiIndex.from_arrays(
        [
            wanted.index.get_level_values("session_id"),
            wanted.index.get_level_values("trial_id"),
        ],
        names=["session_id", "trial_id"],
    ).unique()

    pair_index_all = pd.MultiIndex.from_arrays(
        [
            all_data.index.get_level_values("session_id"),
            all_data.index.get_level_values("trial_id"),
        ],
        names=["session_id", "trial_id"],
    )
    keep_mask = pair_index_all.isin(wanted_pairs)
    data_sub = all_data[keep_mask]

    if data_sub.empty:
        print("No overlap between selected cluster trials and all_data index.")
        return

    if color_vmin is None or color_vmax is None:
        q = data_sub[color_col].quantile([0.05, 0.95]).values
        if color_vmin is None:
            color_vmin = float(q[0])
        if color_vmax is None:
            color_vmax = float(q[1])

    fig = plt.figure(figsize=(10, 4))
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.3)
    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[0, 1], sharey=ax0)
    ax = [ax0, ax1]
    cax = fig.add_subplot(gs[0, 2])

    stop_y, stop_c = [], []
    skip_y, skip_c = [], []
    total_trials = 0
    n_stop = 0
    labeled = False

    grouped = data_sub.groupby(level=["session_id", "trial_id"], sort=False)
    
    if y_lim is None:
        # 99th percentile of y_col across all selected trials for dynamic y-axis scaling
        y_lim_upper = data_sub[y_col].quantile(0.99)
        ylim_lower = data_sub[y_col].quantile(0.01)
        y_lim = (ylim_lower, y_lim_upper)

    for (_, _), trial_data in grouped:
        trial_data = trial_data.reset_index(drop=True)
        if trial_data.empty:
            continue

        y_vals = trial_data[y_col].to_numpy()
        c_vals = trial_data[color_col].to_numpy()
        if np.all(np.isnan(y_vals)) or np.all(np.isnan(c_vals)):
            continue

        total_trials += 1
        is_stop = trial_data.iloc[0]["choice_R1"] > 0
        ax_i = 1 if is_stop else 0
        x = np.arange(len(trial_data))

        if is_stop:
            n_stop += 1
            stop_y.append(y_vals)
            stop_c.append(c_vals)
        else:
            skip_y.append(y_vals)
            skip_c.append(c_vals)

        ax[ax_i].scatter(
            x, y_vals, c=c_vals,
            alpha=0.12, s=6, cmap=cmap,
            vmin=color_vmin, vmax=color_vmax, linewidths=0
        )

        if is_stop and "frame_below_velocity_thr" in trial_data.columns:
            r_time = trial_data.loc[
                trial_data["frame_below_velocity_thr"].astype(bool), y_col
            ].iloc[0:1]
            if not r_time.empty:
                ax[ax_i].scatter(
                    r_time.index, r_time.values,
                    marker="|", s=120, color="k",
                    label="below reward thresh." if not labeled else None
                )
                labeled = True

    if total_trials == 0:
        plt.close(fig)
        print("No trials to plot after filtering.")
        return

    def mean_trace(arr_list):
        min_len = min(len(a) for a in arr_list)
        arr = np.vstack([a[:min_len] for a in arr_list])
        return np.nanmean(arr, axis=0), min_len

    sc_for_cbar = None
    if stop_y:
        stop_y_m, stop_len = mean_trace(stop_y)
        stop_c_m, _ = mean_trace(stop_c)
        sc_for_cbar = ax[1].scatter(
            np.arange(stop_len), stop_y_m,
            c=stop_c_m, s=30, cmap=cmap,
            vmin=color_vmin, vmax=color_vmax
        )

    if skip_y:
        skip_y_m, skip_len = mean_trace(skip_y)
        skip_c_m, _ = mean_trace(skip_c)
        sc_for_cbar = ax[0].scatter(
            np.arange(skip_len), skip_y_m,
            c=skip_c_m, s=30, cmap=cmap,
            vmin=color_vmin, vmax=color_vmax
        )

    xtick_frames = np.array([0, 60, 120])
    xtick_times = xtick_frames - 60
    y_label = y_col.replace("_", " ")

    for axis in ax:
        axis.set_xlim(*x_lim)
        if y_lim is not None:
            axis.set_ylim(*y_lim)
        axis.grid(True, axis="y", alpha=0.3)
        axis.set_xticks(xtick_frames)
        axis.set_xticklabels([f"{t/60:.1f}" for t in xtick_times])
        axis.set_ylabel(y_label)
        axis.axvline(60, color="k", linestyle="--", linewidth=1, alpha=0.6)
        axis.text(
            60, axis.get_ylim()[1] * 0.93, "reward zone entry",
            fontsize=9, ha="center",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
        )
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)

    frac_stop = 100.0 * n_stop / total_trials
    frac_skip = 100.0 - frac_stop
    # ax[0].set_title(f"Skip trials ({frac_skip:.0f}%)", pad=8)
    # ax[1].set_title(f"Stop trials ({frac_stop:.0f}%)", pad=8)
    ax[0].set_xlabel(f"[s], skipped ({frac_skip:.0f}%)")
    ax[1].set_xlabel(f"[s], stopped ({frac_stop:.0f}%)")
    ax[1].legend(loc="best", fontsize=8)

    if sc_for_cbar is not None:
        cbar = fig.colorbar(sc_for_cbar, cax=cax)
        cbar.set_label(color_col.replace("_", " "), rotation=270, labelpad=15)

    fig.suptitle(
        f"All sessions (cluster {target_cluster}, n={total_trials} trials)\n"
        f"y={y_col}, color={color_col}",
        y=0.98, fontsize=11
    )
    fig.tight_layout(rect=[0, 0.02, 1, 0.88])  # reserve top space for suptitle
    plt.show()

In [ ]:
# # Overview iterator: plot every requested y-metric for every cluster
# # Always colors by frame_forward_prop (with fallback to forward_proportion).

# from IPython.display import display, Markdown
# import numpy as np

# # 1) Metrics to iterate (from your list)
# metrics_to_plot = [
#     "frame_raw",
#     "frame_raw_500msMedian",
#     # "frame_yaw",
#     # "frame_pitch",
#     "frame_raw_abs_acc_500msMedian",
#     "frame_RawYawPitch_abs_vel_sum_500msMedian",
#     "frame_YawPitch_abs_vel_sum_500msMedian",
#     "frame_YawPitch_abs_acc_sum_500msMedian",
#     "frame_RawYawPitch_abs_vel_sum",
#     # "trial_id",
#     # "trial_outcome",
#     # "choice_R1",
#     "frame_forward_prop",
#     # "velocity_threshold_at_R1",
#     "forward_vs_rotation_corr",
#     # "frame_below_velocity_thr",
# ]

# # 2) Force color column (your request)
# if "frame_forward_prop" in actions.columns:
#     fixed_color_col = "frame_forward_prop"
# elif "forward_proportion" in actions.columns:
#     fixed_color_col = "forward_proportion"  # fallback if you renamed columns
# else:
#     raise ValueError("Neither 'frame_forward_prop' nor 'forward_proportion' exists in actions.")

# def plot_overview_all_clusters(
#     all_data,
#     hmm_labels_df,
#     metrics,
#     color_col,
#     clusters=None,
#     x_lim=(0, 180),
#     y_lim=None,            # None = auto-scale per plot
#     color_vmin=None,       # None = auto from data quantiles in your function
#     color_vmax=None,
# ):
#     if "cluster_k_id" not in hmm_labels_df.columns:
#         raise ValueError("hmm_labels_df must contain 'cluster_k_id'.")

#     # Choose clusters
#     if clusters is None:
#         clusters = (
#             hmm_labels_df["cluster_k_id"]
#             .dropna()
#             .astype(int)
#             .sort_values()
#             .unique()
#             .tolist()
#         )

#     # Keep only metrics that exist
#     valid_metrics = [m for m in metrics if m in all_data.columns]
#     missing_metrics = [m for m in metrics if m not in all_data.columns]

#     if missing_metrics:
#         print("Skipping missing metrics:")
#         for m in missing_metrics:
#             print(f"  - {m}")

#     if not valid_metrics:
#         raise ValueError("No valid metrics found in all_data.")

#     display(Markdown(f"# All Clusters Overview\nColor metric fixed to: `{color_col}`"))

#     for cluster_id in clusters:
#         if cluster_id == 0:
#             continue  # skip cluster 0 if you want, or remove this condition to include it
#         display(Markdown(f"---\n## Cluster {cluster_id}"))
#         for y_col in valid_metrics:
#             display(Markdown(f"### y: `{y_col}`  |  color: `{color_col}`"))
#             plot_all_sessions_vel_props_fast(
#                 all_data=all_data,
#                 hmm_labels_df=hmm_labels_df,
#                 target_cluster=int(cluster_id),
#                 y_col=y_col,
#                 color_col=color_col,
#                 cmap="RdBu_r",
#                 color_vmin=color_vmin,
#                 color_vmax=color_vmax,
#                 x_lim=x_lim,
#                 y_lim=y_lim,
#             )
#         # break

# # 3) Run
# plot_overview_all_clusters(
#     all_data=actions,
#     hmm_labels_df=hmm_labels,
#     metrics=metrics_to_plot,
#     color_col=fixed_color_col,
#     clusters=None,      # or e.g. [0,1,2,3,4]
#     x_lim=(0, 180),
#     y_lim=None,         # use None for per-metric auto y-range
#     color_vmin=None,    # use None for automatic color limits
#     color_vmax=None,
# )

In [ ]:
plt.close("all")
def plot_session_vel_props(session_data):
    fig = plt.figure(figsize=(10, 4))
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.3)
    ax = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]
    cax = fig.add_subplot(gs[0, 2])
    
    n_choices = 0
    n = len(session_data.index.unique("trial_id"))
    
    stop_choices, stop_choice_c = [], []
    skip_choices, skip_choice_c = [], []
    
    sample_of_trials = session_data.index.unique("trial_id")
    
    labeled = False
    for trial_id in sample_of_trials:
        trial_mask = session_data.index.get_level_values('trial_id') == trial_id
        trial_data = session_data[trial_mask]
        trial_data = trial_data.reset_index(drop=True)
        
        ax_i = 1 if (trial_data.iloc[0]['choice_R1'] %5) >0 else 0
        
        x = np.arange(trial_data.shape[0])
        col = trial_data['frame_forward_prop'].values
        
        if ax_i:
            n_choices += ax_i
            stop_choices.append(trial_data['frame_raw_500msMedian'].values)
            stop_choice_c.append(col)
        else:
            skip_choices.append(trial_data['frame_raw_500msMedian'].values)
            skip_choice_c.append(col)
        
        sc = ax[ax_i].scatter(x, trial_data['frame_raw_500msMedian'].values, c=col, alpha=0.12, s=6, cmap='RdBu_r', vmin=.3, vmax=.7)
        
        if ax_i == 1:
            r_time = trial_data.loc[trial_data['frame_below_velocity_thr'].astype(bool), 'frame_raw_500msMedian'].iloc[0:1]
            ax[ax_i].scatter(r_time.index, r_time.values, marker='|', s=120, color='k', label='below reward thresh.' if not labeled else None)
            labeled = True

    stop_choices = np.stack(stop_choices).mean(0)
    stop_choice_c = np.stack(stop_choice_c).mean(0)
    ax[1].scatter(np.arange(len(stop_choices)), stop_choices, c=stop_choice_c, s=30, cmap='RdBu_r', vmin=.3, vmax=.7)
    
    skip_choices = np.stack(skip_choices).mean(0)
    skip_choice_c = np.stack(skip_choice_c).mean(0)
    sc = ax[0].scatter(np.arange(len(skip_choices)), skip_choices, c=skip_choice_c, s=30, cmap='RdBu_r', vmin=.3, vmax=.7)
    
    # Set xticks to reflect time from t=0 (frame 60 = t=0)
    xtick_frames = np.array([0,60,120])
    xtick_times = xtick_frames-60
    
    for i, axis in enumerate(ax):
        axis.set_xlim(0, 180)
        axis.set_ylim(0, 150)
        axis.grid(True, axis='y', alpha=0.3)
        axis.set_xticks(xtick_frames)
        axis.set_xticklabels([f'{t/60:.1f}' for t in xtick_times])
        axis.set_ylabel('Forward ball velocity (cm/s)')
        axis.axvline(60, color='k', linestyle='--', linewidth=1, alpha=0.6)
        axis.text(60, 140, 'reward zone entry', fontsize=9, ha='center', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # Remove spines
        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)
    
    ax[0].set_title(f"Skip trials ({(n-n_choices)/n*100:.0f}%)")
    ax[0].set_xlabel("[s]")
    ax[1].set_xlabel("[s]")
    ax[1].set_title(f"Stop trials ({n_choices/n*100:.0f}%)")
    ax[1].legend(loc='best', fontsize=8)
    
    # Add colorbar on the dedicated axis
    cbar = plt.colorbar(sc, cax=cax)
    cbar.set_label('Forward Proportion', rotation=270, labelpad=15)
    
    # Add shared x-axis label
    # fig.text(0.5, 0.02, 'Time from R1 entry (frames)', ha='center', fontsize=10)

        
for i, s_id in enumerate(actions.index.unique('session_id')):
    session_data = actions.loc[s_id]
    plot_session_vel_props(session_data)
    plt.suptitle(f"Session {i}, {s_id}", y=0.98)
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.show()
    # break